# Zero-Shot Gambling Comment Detection (Final Optimized v9 - Debugged)

Notebook ini digunakan untuk mendeteksi komentar judi online menggunakan model Zero-Shot Multilingual (`joeddav/xlm-roberta-large-xnli`).

### Perbaikan v9 (Fix Error Empty Sequence):
1. **Filter Data**: Menghapus baris kosong/whitespace yang menyebabkan error `ValueError: You must include at least one label...`.
2. **Strategy**: Tetap menggunakan **Ensemble Strategy** (v8) karena hasilnya paling robust.

### Langkah-langkah:
1. **Runtime > Change runtime type > T4 GPU** (Wajib!)
2. Upload file `comments_labeled_final.csv` ke panel Files.
3. Jalankan cell satu per satu.

In [ ]:
# @title 1. Install Dependencies
!pip install transformers torch pandas tqdm scikit-learn datasets sentencepiece protobuf
print("Dependencies installed!")

In [ ]:
# @title 2. Load Multilingual Model & Prepare Dataset
from transformers import pipeline
from transformers.pipelines.pt_utils import KeyDataset
from datasets import Dataset
import torch
import pandas as pd
from tqdm.auto import tqdm
import os

# Config
INPUT_FILE = "comments_labeled_final.csv"
OUTPUT_FILE = "comments_zeroshot_labeled.csv"
BATCH_SIZE = 16
THRESHOLD = 0.5

# Check GPU
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {device} ({torch.cuda.get_device_name(0) if device==0 else 'CPU'})")

# Load Multilingual Model
classifier = pipeline(
    "zero-shot-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=device,
    batch_size=BATCH_SIZE,
    clean_up_tokenization_spaces=True
)

In [ ]:
# @title 3. 🧪 TEST LABELS (Ensemble Strategy)

# Definisikan list label untuk masing-masing kategori
JUDOL_LABELS = [
    "promosi judi online",
    "iklan situs slot gacor",
    "ajakan main togel"
]

SAFE_LABELS = [
    "komentar biasa",
    "ucapan terima kasih",
    "pertanyaan umum"
]

# Gabungkan semua label untuk di-pass ke model
ALL_LABELS = JUDOL_LABELS + SAFE_LABELS

test_comments = [
    "Main di PULAU777 gacor banget! WD lancar.",  # HARUS JUDOL
    "Jangan lupa gabung ARWANATOTO, pasti JP paus.", # HARUS JUDOL
    "Video ini sangat bermanfaat, terima kasih bang.", # HARUS NORMAL
    "Info loker dong bang daerah Jakarta.", # HARUS NORMAL
    "Situs slot terpercaya hanya di MONA4D, bonus melimpah!" # HARUS JUDOL
]

print("=== TESTING PREDICTION (ENSEMBLE) ===")
for text in test_comments:
    res = classifier(text, ALL_LABELS, multi_label=True)
    scores = dict(zip(res['labels'], res['scores']))
    
    # Ambil score tertinggi dari masing-masing kategori
    max_judol_score = max([scores.get(l, 0) for l in JUDOL_LABELS])
    max_safe_score = max([scores.get(l, 0) for l in SAFE_LABELS])
    
    # Logic: Score Judi harus lebih tinggi dari Score Safe DAN lebih tinggi dari threshold
    is_judol = (max_judol_score > max_safe_score) and (max_judol_score > THRESHOLD)
    
    print(f"Text: {text[:50]}...")
    print(f"Max Judol Score: {max_judol_score:.4f}")
    print(f"Max Safe Score : {max_safe_score:.4f}")
    print(f"Result: {'🔴 JUDOL' if is_judol else '🟢 SAFE'}")
    print("-" * 30)

print("\nJIKA OK, LANJUT STEP 4.")

In [ ]:
# @title 4. Run Batch Classification (Full Dataset)

if not os.path.exists(INPUT_FILE):
    print(f"ERROR: {INPUT_FILE} not found! Upload file first.")
else:
    df = pd.read_csv(INPUT_FILE)
    
    # === FIX ERROR: FILTER EMPTY STRINGS ===
    initial_count = len(df)
    df['comment_text'] = df['comment_text'].fillna("").astype(str)
    
    # Filter out empty or whitespace-only comments
    df_clean = df[df['comment_text'].str.strip().str.len() > 0].copy()
    
    print(f"Loaded {initial_count:,} comments.")
    print(f"Filtered {initial_count - len(df_clean):,} empty/invalid comments.")
    print(f"Processing {len(df_clean):,} valid comments with XLM-R Large...")
    
    if len(df_clean) == 0:
        print("ERROR: No valid comments to process!")
    else:
        hf_dataset = Dataset.from_pandas(df_clean[['comment_text']])
        
        results_label = []
        results_prob = []
        
        # Use batch processing
        for out in tqdm(classifier(KeyDataset(hf_dataset, "comment_text"), 
                                candidate_labels=ALL_LABELS, 
                                multi_label=True, 
                                batch_size=BATCH_SIZE), total=len(hf_dataset)):
            
            scores = dict(zip(out['labels'], out['scores']))
            
            max_judol_score = max([scores.get(l, 0) for l in JUDOL_LABELS])
            max_safe_score = max([scores.get(l, 0) for l in SAFE_LABELS])
            
            label = 1 if (max_judol_score > max_safe_score and max_judol_score > THRESHOLD) else 0
            
            results_label.append(label)
            results_prob.append(max_judol_score)

        # Save Results back to DataFrame
        df_clean['zeroshot_label'] = results_label
        df_clean['zeroshot_prob'] = results_prob

        print("\n=== HASIL ===")
        print(f"Total Processed: {len(df_clean):,}")
        print(f"Judol: {df_clean['zeroshot_label'].sum():,} ({df_clean['zeroshot_label'].mean()*100:.2f}%)")
        print(f"Safe: {(df_clean['zeroshot_label']==0).sum():,}")

        df_clean.to_csv(OUTPUT_FILE, index=False)
        print(f"Saved to {OUTPUT_FILE}")
        
        # Coba download otomatis
        try:
            from google.colab import files
            files.download(OUTPUT_FILE)
        except Exception as e:
            print("Gagal auto-download. Silakan download manual dari panel Files.")